# Triarchy — Portfolio Analysis

Multi-year walk-forward analysis across 2022–2025.

This notebook aggregates the per-year backtest results produced by `triarchy run --years 2022 2023 2024 2025` and presents a unified portfolio view: cross-year metrics, equity curve, drawdown, monthly performance, and breakdown by regime.

**Methodology:** see [`docs/walk_forward.md`](../docs/walk_forward.md). The same configuration is used for every year — no parameters are tuned per year.

> **Note**: Numbers in this notebook are regenerated each time you run `triarchy run`. The plots embedded in the README may reflect a previous run.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from triarchy.backtest.metrics import build_report, BARS_PER_YEAR_1H
from triarchy.backtest.plots import (
    plot_equity_curve, plot_drawdown, plot_monthly_returns_heatmap,
    plot_r_distribution, plot_performance_by_regime,
)

sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

RESULTS_ROOT = Path('../BACKTEST_RESULTS')
YEARS = [2022, 2023, 2024, 2025]

## 1. Load per-year results

Each year writes a `summary.csv`, `trades.csv`, and `equity_curve.csv` to its results directory. We load all years and concatenate.

In [ ]:
def load_year(year):
    base = RESULTS_ROOT / str(year)
    if not base.exists():
        return None
    summary = pd.read_csv(base / 'summary.csv').iloc[0].to_dict() if (base / 'summary.csv').exists() else None
    trades = pd.read_csv(base / 'trades.csv') if (base / 'trades.csv').exists() else pd.DataFrame()
    eq = pd.read_csv(base / 'equity_curve.csv', index_col=0, parse_dates=True)['equity'] if (base / 'equity_curve.csv').exists() else pd.Series(dtype=float)
    return {'year': year, 'summary': summary, 'trades': trades, 'equity': eq}

loaded = {y: load_year(y) for y in YEARS}
loaded = {y: d for y, d in loaded.items() if d is not None and d['summary'] is not None}
list(loaded.keys())

## 2. Cross-year summary table

The single most important table for a recruiter. Year-by-year metrics tell the story of robustness.

In [ ]:
rows = []
for year, data in loaded.items():
    s = data['summary']
    rows.append({
        'Year': year,
        'Trades': int(s['n_trades']),
        'Win rate': f"{s['win_rate']*100:.1f}%",
        'Avg R': f"{s['avg_r']:.2f}",
        'Profit factor': f"{s['profit_factor']:.2f}" if s['profit_factor'] != float('inf') else '∞',
        'Total return': f"{s['total_return_pct']*100:.1f}%",
        'CAGR': f"{s['cagr']*100:.1f}%",
        'Sharpe': f"{s['sharpe']:.2f}",
        'Sortino': f"{s['sortino']:.2f}",
        'Max DD': f"{s['max_drawdown_pct']*100:.1f}%",
        'Calmar': f"{s['calmar']:.2f}",
    })
summary_df = pd.DataFrame(rows).set_index('Year')
summary_df

## 3. Combined multi-year equity curve

Each year is reset to the same starting equity. Stitching them together gives a visual of cross-year compounding.

In [ ]:
starting_equity = float(list(loaded.values())[0]['summary']['starting_equity'])

# Stitch: each year contributes its PnL increments anchored to the prior year's end
stitched_increments = []
for year in sorted(loaded.keys()):
    eq = loaded[year]['equity']
    if eq.empty:
        continue
    inc = eq.diff().fillna(eq.iloc[0] - starting_equity)
    stitched_increments.append(inc)

stitched = pd.concat(stitched_increments).sort_index()
portfolio_eq = starting_equity + stitched.cumsum()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(portfolio_eq.index, portfolio_eq.values, linewidth=1.4, color='#1f77b4')
ax.fill_between(portfolio_eq.index, portfolio_eq.values, starting_equity, alpha=0.08, color='#1f77b4')
ax.axhline(starting_equity, color='gray', linestyle='--', linewidth=0.8)
for year in sorted(loaded.keys())[1:]:
    ax.axvline(pd.Timestamp(f'{year}-01-01', tz='UTC'), color='gray', linewidth=0.5, alpha=0.5)
ax.set_title('Multi-year equity curve (walk-forward)', fontsize=14, fontweight='bold')
ax.set_ylabel('Equity ($)')
fig.tight_layout()
plt.show()

## 4. Trade quality by regime (across all years)

If the system is doing what it's supposed to, average R should be highest in TRENDING regimes (where TREND_FOLLOW playbooks fire) and the engine should refuse to trade in CRASH.

In [ ]:
all_trades = pd.concat(
    [d['trades'] for d in loaded.values() if not d['trades'].empty],
    ignore_index=True,
)
if not all_trades.empty:
    by_regime = all_trades.groupby('regime').agg(
        trades=('pnl', 'count'),
        avg_r=('r_mult', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean()),
        total_pnl=('pnl', 'sum'),
    ).round(3)
    display(by_regime)
else:
    print('No trades to analyze yet.')

## 5. Trade quality by playbook

In [ ]:
if not all_trades.empty:
    by_playbook = all_trades.groupby('playbook').agg(
        trades=('pnl', 'count'),
        avg_r=('r_mult', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean()),
        total_pnl=('pnl', 'sum'),
    ).round(3)
    display(by_playbook)

## 6. Trade quality by symbol

In [ ]:
if not all_trades.empty:
    by_symbol = all_trades.groupby('symbol').agg(
        trades=('pnl', 'count'),
        avg_r=('r_mult', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean()),
        total_pnl=('pnl', 'sum'),
    ).round(3).sort_values('total_pnl', ascending=False)
    display(by_symbol)

## 7. R-multiple distribution

A healthy systematic strategy has a positively-skewed R distribution: many small losses (around -1R) capped by stops, fewer but larger wins (2R or beyond).

In [ ]:
if not all_trades.empty:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.hist(all_trades['r_mult'], bins=40, color='#2ca02c', alpha=0.75, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(all_trades['r_mult'].mean(), color='#d62728', linewidth=1.5,
               label=f"Mean R = {all_trades['r_mult'].mean():.2f}")
    ax.set_title('R-multiple distribution — all years combined', fontsize=13, fontweight='bold')
    ax.set_xlabel('R')
    ax.legend()
    fig.tight_layout()
    plt.show()

## 8. Robustness check — Sharpe stability across years

The single most important robustness signal. A strategy whose Sharpe swings wildly across years is overfit. A strategy whose Sharpe stays in a reasonable band is robust.

In [ ]:
sharpes = {y: d['summary']['sharpe'] for y, d in loaded.items()}
sortinos = {y: d['summary']['sortino'] for y, d in loaded.items()}
ddowns = {y: d['summary']['max_drawdown_pct']*100 for y, d in loaded.items()}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, data, color) in zip(axes, [
    ('Sharpe', sharpes, '#1f77b4'),
    ('Sortino', sortinos, '#2ca02c'),
    ('Max drawdown (%)', ddowns, '#d62728'),
]):
    bars = ax.bar(list(data.keys()), list(data.values()), color=color, alpha=0.75, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.2f}', ha='center',
                va='bottom' if h >= 0 else 'top', fontsize=9)
fig.suptitle('Year-by-year stability', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

## 9. Discussion

Things to look at when interpreting these results:

1. **Cross-year stability** — do Sharpe/Sortino stay in a reasonable band? If one year carries the equity curve while others lose money, the system is fragile.
2. **Regime alignment** — TREND_FOLLOW trades should win disproportionately in TRENDING regimes. A trend-follower that wins in RANGING years is doing something it wasn't designed for (lucky, not robust).
3. **Drawdown bound** — across very different market conditions, max drawdown should stay bounded (the strategy has built-in risk gating: CRASH regime → RISK_MODE=OFF).
4. **Number of trades** — should scale with opportunity. Trending years naturally produce more trade setups than choppy ones.

## Honest caveats

- 4 years of crypto data is **not** enough for statistical robustness — we'd want 10+ years across multiple assets and regimes. Crypto's history is short.
- Slippage is modeled at 1 bp; in stressed conditions (CRASH events), real slippage can be 10–50 bps.
- All symbols are equal-weighted in the portfolio aggregation. Real portfolio construction should account for correlation and volatility targeting.
- The strategy's parameters were chosen from first principles (standard EMA/ATR/ADX values), not optimized — but the *structural choices* (3-layer hierarchy, gating logic, R-multiple targets) themselves embody opinions that could be wrong.

These caveats are part of why this project includes an **agentic overlay roadmap**: an LLM-driven Macro Agent can supplement rule-based regime detection with unstructured context (news, macro events) that pure indicators miss.